# Hedonic Pricing — Thesis Figures
Publication-ready charts for the thesis. Run `hedonic_regression.ipynb` first to generate the tables.

**Figures:**
1. Price distribution by brand tier (violin plots)
2. Coefficient forest plot (pooled regression)
3. Brand premium: marked vs. final price
4. Premium heatmap across categories × attributes
5. Promotion penetration vs. brand premium (scatter)

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.formula.api as smf

# Colorblind-safe palette (Wong 2011)
COLORS = {
    'blue':   '#0072B2',
    'orange': '#E69F00',
    'green':  '#009E73',
    'red':    '#D55E00',
    'purple': '#CC79A7',
    'sky':    '#56B4E9',
    'yellow': '#F0E442',
    'black':  '#000000',
}

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

from pathlib import Path
from src.utils import PROJECT_ROOT
OUT = PROJECT_ROOT / 'output'

In [ ]:
df = pd.read_csv(OUT / 'product_features.csv')
print(f"{len(df)} products loaded")

# Regression sample (same filter as hedonic_regression.ipynb)
reg_df = df.dropna(subset=['ln_price_per_unit', 'ln_pack_size', 'subcategory']).copy()
reg_df = reg_df[np.isfinite(reg_df['ln_price_per_unit'])]
lo, hi = reg_df['ln_price_per_unit'].quantile([0.01, 0.99])
reg_df = reg_df[(reg_df['ln_price_per_unit'] >= lo) & (reg_df['ln_price_per_unit'] <= hi)]
print(f"Regression sample: {len(reg_df)} products")

## Figure 1: Price Distribution by Brand Tier
Violin plots comparing branded vs. generic unit prices across categories.

In [ ]:
# Focus on categories with enough branded + generic products
cat_order = (
    reg_df.groupby('parent_category')['avg_final_price'].median()
    .sort_values(ascending=False).index.tolist()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, col, title in zip(
    axes,
    ['ln_price_per_unit', 'ln_marked_price_per_unit'],
    ['Final Price (checkout)', 'Marked Price (sticker)'],
):
    branded   = reg_df[reg_df['is_branded'] == 1]
    unbranded = reg_df[reg_df['is_branded'] == 0]

    # Box plot comparison
    data_plot = [
        branded[col].dropna().values,
        unbranded[col].dropna().values,
    ]
    vp = ax.violinplot(data_plot, positions=[1, 2], showmedians=True, widths=0.6)
    vp['bodies'][0].set_facecolor(COLORS['blue'])
    vp['bodies'][1].set_facecolor(COLORS['orange'])
    for pc in ['cmedians', 'cmins', 'cmaxes', 'cbars']:
        vp[pc].set_color('black')
        vp[pc].set_linewidth(1)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Branded', 'Generic'])
    ax.set_ylabel('ln(price per 100g/ml)')
    ax.set_title(title)
    ax.text(0.05, 0.95, f"n={len(branded)} / {len(unbranded)}",
            transform=ax.transAxes, va='top', fontsize=9, color='gray')

fig.suptitle('Figure 1: Unit Price Distribution — Branded vs. Generic', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'thesis_fig1_brand_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Fig 1")

## Figure 2: Coefficient Forest Plot
All key regressors from the pooled OLS, with 95% CIs. Cluster-robust SEs.

In [ ]:
FORMULA = (
    'ln_price_per_unit ~ '
    'is_branded + is_import + is_house_brand + '
    'ln_pack_size + pack_count + '
    'has_health_claim + has_freshness_claim + '
    'name_length + C(subcategory)'
)

model_final  = smf.ols(FORMULA, data=reg_df).fit(
    cov_type='cluster', cov_kwds={'groups': reg_df['subcategory'].astype(str)})
model_marked = smf.ols(FORMULA.replace('ln_price_per_unit', 'ln_marked_price_per_unit'),
                        data=reg_df).fit(
    cov_type='cluster', cov_kwds={'groups': reg_df['subcategory'].astype(str)})

KEY_VARS = [
    ('is_branded',          'Branded product'),
    ('is_import',           'Imported origin'),
    ('is_house_brand',      'House brand (WinEco)'),
    ('ln_pack_size',        'ln(pack size)'),
    ('pack_count',          'Pack count'),
    ('has_health_claim',    'Health claim'),
    ('has_freshness_claim', 'Freshness claim'),
    ('name_length',         'Name length (tokens)'),
]

fig, ax = plt.subplots(figsize=(9, 5))

y_pos = np.arange(len(KEY_VARS))
offset = 0.18  # vertical offset between final/marked

for i, (mod, label, color) in enumerate([
    (model_marked, 'Marked price', COLORS['orange']),
    (model_final,  'Final price',  COLORS['blue']),
]):
    coefs = [mod.params.get(v, np.nan) for v, _ in KEY_VARS]
    cis   = [mod.conf_int().loc[v].values if v in mod.params else [np.nan, np.nan]
             for v, _ in KEY_VARS]
    lo_ci = [c[0] for c in cis]
    hi_ci = [c[1] for c in cis]

    y = y_pos + (i - 0.5) * offset
    ax.errorbar(
        coefs, y,
        xerr=[np.array(coefs) - np.array(lo_ci), np.array(hi_ci) - np.array(coefs)],
        fmt='o', color=color, capsize=4, markersize=6, label=label,
        linewidth=1.5,
    )

ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels([label for _, label in KEY_VARS])
ax.set_xlabel('Coefficient (log-point premium)')
ax.set_title('Figure 2: Hedonic Price Premiums (Pooled OLS, Cluster-Robust 95% CI)',
             fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(ax.get_xlim())  # keep symmetric

plt.tight_layout()
plt.savefig(OUT / 'thesis_fig2_forest_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Fig 2")

## Figure 3: Brand Premium — Marked vs. Final Price
Scatter: each point = one category. X = brand premium in marked price, Y = brand premium in final price.
Points below the 45° line = promotions partially erode the brand premium.

In [ ]:
cat_results_final  = {}
cat_results_marked = {}

cat_counts = reg_df.groupby('parent_category').size()
eligible = cat_counts[cat_counts >= 20].index.tolist()

SIMPLE_F = (
    'is_branded + is_import + is_house_brand + '
    'ln_pack_size + pack_count + has_health_claim + has_freshness_claim + name_length'
)

for cat in eligible:
    sub = reg_df[reg_df['parent_category'] == cat]
    fe = ' + C(subcategory)' if sub['subcategory'].nunique() > 1 else ''
    try:
        mf = smf.ols(f'ln_price_per_unit ~ {SIMPLE_F}{fe}', data=sub).fit()
        mm = smf.ols(f'ln_marked_price_per_unit ~ {SIMPLE_F}{fe}', data=sub).fit()
        cat_results_final[cat]  = mf.params.get('is_branded', np.nan)
        cat_results_marked[cat] = mm.params.get('is_branded', np.nan)
    except Exception:
        pass

cats  = list(cat_results_final.keys())
x_val = [cat_results_marked[c] for c in cats]
y_val = [cat_results_final[c]  for c in cats]

fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(x_val, y_val, color=COLORS['blue'], s=80, zorder=3)

# 45° line
lim = [min(x_val + y_val) - 0.05, max(x_val + y_val) + 0.05]
ax.plot(lim, lim, 'k--', linewidth=0.8, alpha=0.5, label='45° (no erosion)')

for cat, x, y in zip(cats, x_val, y_val):
    short = cat.split()[0] if len(cat) > 15 else cat
    ax.annotate(short, (x, y), textcoords='offset points', xytext=(6, 3), fontsize=8)

ax.set_xlabel('Brand premium in marked price (ln-points)')
ax.set_ylabel('Brand premium in final price (ln-points)')
ax.set_title('Figure 3: Promotional Erosion of Brand Premiums by Category',
             fontweight='bold')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT / 'thesis_fig3_brand_premium_marked_vs_final.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Fig 3")

## Figure 4: Premium Heatmap — Attributes × Categories

In [ ]:
ATTR_VARS = ['is_branded', 'is_import', 'is_house_brand', 'has_health_claim', 'has_freshness_claim']
ATTR_LABELS = ['Branded', 'Imported', 'House brand', 'Health claim', 'Freshness claim']

heatmap_data = pd.DataFrame(index=eligible, columns=ATTR_VARS, dtype=float)

for cat in eligible:
    sub = reg_df[reg_df['parent_category'] == cat]
    fe = ' + C(subcategory)' if sub['subcategory'].nunique() > 1 else ''
    try:
        m = smf.ols(f'ln_price_per_unit ~ {SIMPLE_F}{fe}', data=sub).fit()
        for v in ATTR_VARS:
            heatmap_data.loc[cat, v] = m.params.get(v, np.nan)
    except Exception:
        pass

heatmap_data = heatmap_data.astype(float)

fig, ax = plt.subplots(figsize=(10, max(4, len(eligible) * 0.5 + 1)))

vmax = heatmap_data.abs().max().max()
im = ax.imshow(
    heatmap_data.values,
    cmap='RdBu_r', aspect='auto',
    vmin=-vmax, vmax=vmax,
)
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Coefficient (ln-points)')

# Annotate cells
for i in range(len(eligible)):
    for j in range(len(ATTR_VARS)):
        val = heatmap_data.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if abs(val) > vmax * 0.6 else 'black')

ax.set_xticks(range(len(ATTR_VARS)))
ax.set_xticklabels(ATTR_LABELS, rotation=30, ha='right')
ax.set_yticks(range(len(eligible)))
ax.set_yticklabels([c[:20] for c in eligible], fontsize=9)
ax.set_title('Figure 4: Price Premium Heatmap (ln_price_per_unit DV)', fontweight='bold')

plt.tight_layout()
plt.savefig(OUT / 'thesis_fig4_premium_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Fig 4")

## Figure 5: Promotion Penetration vs. Brand Premium
Cross-category scatter: does higher promotion penetration correlate with higher brand premium?
Tests the 'promotional pricing paradox' hypothesis.

In [ ]:
# Per-category: avg promo_rate and brand premium in marked price
cat_promo = df.groupby('parent_category')['promo_rate'].mean()

# Use marked-price brand premium (shows full sticker premium before discounts)
brand_premiums_marked = {}
for cat in eligible:
    sub = reg_df[reg_df['parent_category'] == cat]
    fe  = ' + C(subcategory)' if sub['subcategory'].nunique() > 1 else ''
    try:
        mm = smf.ols(f'ln_marked_price_per_unit ~ {SIMPLE_F}{fe}', data=sub).fit()
        brand_premiums_marked[cat] = mm.params.get('is_branded', np.nan)
    except Exception:
        pass

plot_cats   = [c for c in eligible if c in brand_premiums_marked and c in cat_promo.index]
x_promo     = [cat_promo[c] * 100 for c in plot_cats]  # as %
y_premium   = [brand_premiums_marked[c] for c in plot_cats]

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(x_promo, y_premium, color=COLORS['blue'], s=90, zorder=3)

for cat, x, y in zip(plot_cats, x_promo, y_premium):
    short = cat.split()[0] if len(cat) > 15 else cat
    ax.annotate(short, (x, y), textcoords='offset points', xytext=(6, 3), fontsize=8)

# Regression line
if len(x_promo) >= 3:
    m, b = np.polyfit(x_promo, y_premium, 1)
    x_line = np.linspace(min(x_promo), max(x_promo), 50)
    ax.plot(x_line, m * x_line + b, '--', color=COLORS['red'], linewidth=1.2,
            label=f'OLS fit (slope={m:.3f})')
    r, p = stats.pearsonr(x_promo, y_premium)
    ax.text(0.05, 0.95, f'r = {r:.2f}, p = {p:.3f}',
            transform=ax.transAxes, va='top', fontsize=9)

ax.set_xlabel('Promotion penetration rate (% products on sale)')
ax.set_ylabel('Brand premium in marked price (ln-points)')
ax.set_title('Figure 5: Promotional Pricing Paradox — Penetration vs. Brand Premium',
             fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT / 'thesis_fig5_promo_vs_brand_premium.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Fig 5")

In [ ]:
print("All thesis figures saved to output/")
import os
figs = sorted(f for f in os.listdir('../output') if f.startswith('thesis_fig'))
for f in figs:
    print(f"  {f}")